# 📊 Dataset Analysis

## Thesis Title

**Bidirectional Neural Machine Translation Between
Standard Bangla and Chatgaya Dialect Using
Pretrained Transformer Models**

---

### Objective

Before training the translation model, it is important to understand the quality and characteristics of the dataset.

This notebook analyzes the manually created canonical dataset and provides useful statistics that will help improve the dataset before model training.

The analysis includes:

- Dataset overview
- Missing values
- Duplicate detection
- Sentence statistics
- Vocabulary statistics
- Bangla word frequency
- Chatgaya word frequency
- Rare word detection
- Context-sensitive word coverage
- Dataset health report

---

**Author:** Mostafa Al Moin

In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

# pandas:
# Used for loading and manipulating the dataset.

import pandas as pd


# re (Regular Expression):
# Used for removing punctuation before word tokenization.

import re


# Counter:
# Used to count the frequency of each word.

from collections import Counter

In [2]:
# ==========================================================
# Load Canonical Dataset
# ==========================================================

# Read the canonical dataset from the Excel file.
# The dataset contains two columns:
#
# Bangla     -> Standard Bangla sentence
# Chatgaya   -> Corresponding Chatgaya translation

df = pd.read_excel("canonical.xlsx")


# Rename the columns to maintain
# a consistent naming convention.

df.columns = ["Bangla", "Chatgaya"]


# Display the first five rows
# to verify that the dataset
# has been loaded correctly.

df.head()

,Bangla,Chatgaya
0,আসসালামু আলাইকুম ম্যাম,আসসালামু আলাইকুম ম্যাম
1,আসসালামু আলাইকুম স্যার,আসসালামু আলাইকুম স্যার
2,আসসালামু আলাইকুম ভাই,আসসালামু আলাইকুম বদ্দা
3,নমস্কার ম্যাম,নমস্কার ম্যাম
4,কেমন আছেন?,কেন আছন?


In [3]:
# ==========================================================
# Dataset Overview
# ==========================================================

# This section provides a general overview
# of the canonical dataset.

print("="*60)
print("DATASET OVERVIEW")
print("="*60)

print(f"Total Sentence Pairs : {len(df)}")

print(f"Number of Columns    : {len(df.columns)}")

print()

print("Column Names:")

for column in df.columns:
    print(f"• {column}")

DATASET OVERVIEW
Total Sentence Pairs : 1810
Number of Columns    : 2

Column Names:
• Bangla
• Chatgaya


In [4]:
# ==========================================================
# Dataset Information
# ==========================================================

# Display detailed information about the dataset,
# including column names, data types,
# and non-null values.

print("="*60)
print("DATASET INFORMATION")
print("="*60)

df.info()

DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1810 entries, 0 to 1809
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Bangla    1810 non-null   object
 1   Chatgaya  1810 non-null   object
dtypes: object(2)
memory usage: 28.4+ KB


In [5]:
# ==========================================================
# Missing Value Analysis
# ==========================================================

# Missing values can negatively affect
# model training.
#
# Therefore, we check whether any
# sentence pair is incomplete.

print("="*60)
print("MISSING VALUES")
print("="*60)

df.isnull().sum()

MISSING VALUES


,0
Bangla,0
Chatgaya,0


In [6]:
# ==========================================================
# Duplicate Analysis
# ==========================================================

# Duplicate sentence pairs may introduce
# unnecessary bias into the model.
#
# Here we check:
#
# 1. Duplicate Bangla sentences
# 2. Duplicate Chatgaya sentences
# 3. Duplicate sentence pairs

duplicate_bangla = df["Bangla"].duplicated().sum()

duplicate_chatgaya = df["Chatgaya"].duplicated().sum()

duplicate_pairs = df.duplicated().sum()


print("="*60)
print("DUPLICATE ANALYSIS")
print("="*60)

print(f"Duplicate Bangla Sentences   : {duplicate_bangla}")

print(f"Duplicate Chatgaya Sentences : {duplicate_chatgaya}")

print(f"Duplicate Sentence Pairs     : {duplicate_pairs}")

DUPLICATE ANALYSIS
Duplicate Bangla Sentences   : 0
Duplicate Chatgaya Sentences : 0
Duplicate Sentence Pairs     : 0


## Interpretation

At this stage we verified the basic quality of the dataset.

A good translation dataset should ideally have:

- No missing values
- No accidental duplicate sentence pairs
- Correct column names
- Consistent formatting

If problems are detected here, they should be fixed before proceeding to model training.

# 📌 Bangla Word Frequency Analysis

## Objective

In this section, we calculate the frequency of every Bangla word in the canonical dataset.

This analysis helps us to:

- Identify the most frequently used words.
- Detect rare words with very low coverage.
- Find important words that require more training examples.
- Improve the dataset before model training.

In [7]:
# ==========================================================
# Tokenization Function
# ==========================================================

# Before counting words, we need to split each sentence
# into individual words (tokens).
#
# We also remove common punctuation so that words like
# "আজ," and "আজ" are counted as the same word.

def tokenize(sentence):

    sentence = str(sentence)

    # Convert to lowercase (mainly useful for English words)
    sentence = sentence.lower()

    # Remove common punctuation
    sentence = re.sub(r"[।,!?;:\"'()\-\[\]{}]", " ", sentence)

    # Split sentence into words
    words = sentence.split()

    return words

In [8]:
# ==========================================================
# Bangla Word Frequency
# ==========================================================

# Count how many times each Bangla word appears
# in the canonical dataset.

bangla_counter = Counter()

for sentence in df["Bangla"]:

    words = tokenize(sentence)

    bangla_counter.update(words)

# Convert the frequency dictionary into a DataFrame
bangla_frequency = pd.DataFrame(
    bangla_counter.items(),
    columns=["Word", "Frequency"]
)

# Sort words by frequency (highest first)
bangla_frequency = bangla_frequency.sort_values(
    by="Frequency",
    ascending=False
).reset_index(drop=True)

# Add ranking
bangla_frequency.insert(
    0,
    "Rank",
    range(1, len(bangla_frequency)+1)
)

# Display top 20 words
bangla_frequency.head(20)

,Rank,Word,Frequency
0,1,না,453
1,2,আমার,341
2,3,আমি,241
3,4,অনেক,193
4,5,করে,177
5,6,হবে,174
6,7,থেকে,165
7,8,একটু,136
8,9,বেশি,135
9,10,কি,131


In [9]:
# ==========================================================
# Save Bangla Word Frequency
# ==========================================================

# Save the frequency table so it can be analyzed later
# using Excel.

bangla_frequency.to_excel(
    "bangla_word_frequency.xlsx",
    index=False
)

print("Bangla word frequency saved successfully.")

Bangla word frequency saved successfully.


In [10]:
# ==========================================================
# Search Word Frequency
# ==========================================================

# This function searches for a specific Bangla word
# and returns its frequency in the dataset.

def search_word(word):

    result = bangla_frequency[
        bangla_frequency["Word"] == word
    ]

    if result.empty:

        print(f"'{word}' was not found in the dataset.")

    else:

        print(result)

In [11]:
# Example

search_word("সাহায্য")

     Rank     Word  Frequency
620   621  সাহায্য          4


In [12]:
# ==========================================================
# Search Multiple Words
# ==========================================================

# Search multiple words at once.

words = [
    "ঘুরতে",
    "সাহায্য",
    "অপেক্ষা",
    "সিদ্ধান্ত",
    "কারণ",
    "এখন"
]

for word in words:

    search_word(word)

    print("-"*50)

     Rank   Word  Frequency
341   342  ঘুরতে          8
--------------------------------------------------
     Rank     Word  Frequency
620   621  সাহায্য          4
--------------------------------------------------
     Rank     Word  Frequency
251   252  অপেক্ষা         11
--------------------------------------------------
     Rank       Word  Frequency
435   436  সিদ্ধান্ত          6
--------------------------------------------------
     Rank  Word  Frequency
411   412  কারণ          7
--------------------------------------------------
    Rank Word  Frequency
12    13  এখন        101
--------------------------------------------------


## Interpretation

Words that appear only one or two times may not provide enough contextual information for the model to learn different grammatical patterns.

Such words should be reviewed manually, and additional sentence pairs can be added if necessary to improve dataset coverage.

# 📌 Chatgaya Word Frequency Analysis

## Objective

In this section, we calculate the frequency of every Chatgaya word in the canonical dataset.

This analysis helps us to:

- Understand the vocabulary distribution of the Chatgaya corpus.
- Identify the most frequently used Chatgaya words.
- Detect low-frequency words.
- Compare Bangla and Chatgaya vocabulary coverage.

In [13]:
# ==========================================================
# Chatgaya Word Frequency
# ==========================================================

# Count how many times each Chatgaya word appears
# in the canonical dataset.

chatgaya_counter = Counter()

for sentence in df["Chatgaya"]:

    words = tokenize(sentence)

    chatgaya_counter.update(words)

# Convert the frequency dictionary into a DataFrame
chatgaya_frequency = pd.DataFrame(
    chatgaya_counter.items(),
    columns=["Word", "Frequency"]
)

# Sort by highest frequency
chatgaya_frequency = chatgaya_frequency.sort_values(
    by="Frequency",
    ascending=False
).reset_index(drop=True)

# Add ranking
chatgaya_frequency.insert(
    0,
    "Rank",
    range(1, len(chatgaya_frequency)+1)
)

chatgaya_frequency.head(20)

,Rank,Word,Frequency
0,1,ন,432
1,2,আঁর,246
2,3,আঁই,241
3,4,বত,214
4,5,গরি,178
5,6,অই,166
6,7,কি,146
7,8,এক্কানা,138
8,9,বেশি,138
9,10,ভালা,128


In [14]:
# ==========================================================
# Save Chatgaya Word Frequency
# ==========================================================

# Save the Chatgaya word frequency table
# for later analysis.

chatgaya_frequency.to_excel(
    "chatgaya_word_frequency.xlsx",
    index=False
)

print("Chatgaya word frequency saved successfully.")

Chatgaya word frequency saved successfully.


# 📌 Rare Word Analysis

## Objective

Words with very low frequency provide fewer learning examples for the model.

These words may require additional sentence pairs to improve their contextual coverage before model training.

In [17]:
# ==========================================================
# Rare Word Threshold
# ==========================================================

# Words appearing less than or equal to this threshold
# will be considered as rare words.

RARE_WORD_THRESHOLD = 3

In [18]:
# ==========================================================
# Rare Bangla Words
# ==========================================================

# Extract Bangla words whose frequency is less than
# or equal to the threshold.

rare_bangla = bangla_frequency[
    bangla_frequency["Frequency"] <= RARE_WORD_THRESHOLD
].copy()

print(f"Total Rare Bangla Words: {len(rare_bangla)}")

rare_bangla.head(20)

Total Rare Bangla Words: 2181


,Rank,Word,Frequency
769,770,নমস্কার,3
770,771,হ্যালো,3
771,772,সাহস,3
772,773,গাছ,3
773,774,দোকানদার,3
774,775,হলেই,3
775,776,কোথা,3
776,777,বিড়ালটা,3
777,778,অসুখ,3
778,779,গাড়ি,3


In [15]:
# ==========================================================
# Rare Chatgaya Words
# ==========================================================

# Show Chatgaya words whose frequency is less than or equal to 3.

rare_chatgaya = chatgaya_frequency[
    chatgaya_frequency["Frequency"] <= RARE_WORD_THRESHOLD
]

print(f"Total Rare Chatgaya Words : {len(rare_chatgaya)}")

rare_chatgaya.head(30)

Total Rare Chatgaya Words : 2243


,Rank,Word,Frequency
771,772,মাজন,3
772,773,দরজার,3
773,774,হাতা,3
774,775,ইবেত্তুন,3
775,776,পর্যন্ত,3
776,777,টানি,3
777,778,পর্দাগান,3
778,779,সাদা,3
779,780,হাশি,3
780,781,হ্যালো,3


In [19]:
# ==========================================================
# Save Rare Word Lists
# ==========================================================

rare_bangla.to_excel(
    "rare_bangla_words.xlsx",
    index=False
)

rare_chatgaya.to_excel(
    "rare_chatgaya_words.xlsx",
    index=False
)

print("Rare word reports saved successfully.")

Rare word reports saved successfully.


# 📌 Context-sensitive Word Coverage

## Objective

Some Bangla words require multiple contexts for the model to learn their correct Chatgaya translation.

This section measures how many times those important words appear in the canonical dataset.

Low-frequency context-sensitive words should be considered for adding more sentence pairs.

In [20]:
# ==========================================================
# Context-sensitive Word Coverage
# ==========================================================

# List of important words to monitor.
# You can modify this list anytime.

context_words = [
    "ঘুরতে",
    "সাহায্য",
    "অপেক্ষা",
    "সিদ্ধান্ত",
    "লাগে",
    "হয়",
    "মনে",
    "পারি",
    "হবে",
]

coverage = bangla_frequency[
    bangla_frequency["Word"].isin(context_words)
]

coverage

,Rank,Word,Frequency
5,6,হবে,174
24,25,মনে,76
59,60,লাগে,40
150,151,হয়,20
173,174,পারি,17
251,252,অপেক্ষা,11
341,342,ঘুরতে,8
435,436,সিদ্ধান্ত,6
620,621,সাহায্য,4


In [21]:
# ==========================================================
# Save Context Coverage Report
# ==========================================================

coverage.to_excel(
    "context_word_coverage.xlsx",
    index=False
)

print("Context word coverage report saved successfully.")

Context word coverage report saved successfully.


# 📌 Dataset Health Report

## Objective

Finally, summarize the overall quality of the canonical dataset before model training.

In [22]:
# ==========================================================
# Dataset Health Report
# ==========================================================

print("=" * 60)
print("DATASET HEALTH REPORT")
print("=" * 60)

print(f"Total Sentence Pairs          : {len(df)}")

print(f"Bangla Vocabulary Size        : {len(bangla_frequency)}")

print(f"Chatgaya Vocabulary Size      : {len(chatgaya_frequency)}")

print(f"Duplicate Sentence Pairs      : {duplicate_pairs}")

print(f"Missing Bangla Values         : {df['Bangla'].isnull().sum()}")

print(f"Missing Chatgaya Values       : {df['Chatgaya'].isnull().sum()}")

print(f"Rare Bangla Words (<=3)       : {len(rare_bangla)}")

print(f"Rare Chatgaya Words (<=3)     : {len(rare_chatgaya)}")

print("=" * 60)

DATASET HEALTH REPORT
Total Sentence Pairs          : 1810
Bangla Vocabulary Size        : 2950
Chatgaya Vocabulary Size      : 3014
Duplicate Sentence Pairs      : 0
Missing Bangla Values         : 0
Missing Chatgaya Values       : 0
Rare Bangla Words (<=3)       : 2181
Rare Chatgaya Words (<=3)     : 2243
